<a href="https://colab.research.google.com/github/glebas2310/Eyecar/blob/master/YOLOv4_tiny_Darknet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Обучение YOLOv4-tiny (Darknet) на датасете из Roboflow
Перед запуском убедитесь, что включен GPU: `Среда выполнения` -> `Сменить среду выполнения` -> `T4 GPU`       
Укажите код на скачивание датасета из `Roboflow` в ячейке 2.

In [ ]:
%cd /content
!rm -rf darknet
!git clone https://github.com/AlexeyAB/darknet.git
%cd /content/darknet

# Включаем GPU, CUDNN
!sed -i 's/GPU=0/GPU=1/' Makefile
!sed -i 's/CUDNN=0/CUDNN=1/' Makefile
!sed -i 's/CUDNN_HALF=0/CUDNN_HALF=1/' Makefile

# Очищаем все стандартные архитектуры и прописываем строго Tesla T4 (compute_75)
!sed -i 's/ARCH= -gencode/# ARCH= -gencode/g' Makefile
!sed -i 's/ARCH:= -gencode/# ARCH:= -gencode/g' Makefile
!echo 'ARCH= -gencode arch=compute_75,code=[sm_75,compute_75]' >> Makefile

# Запускаем сборку и скачиваем веса
!make -j$(nproc)
!wget -nc -q https://github.com/AlexeyAB/darknet/releases/download/darknet_yolo_v4_pre/yolov4-tiny.conv.29
!ls -lh darknet

In [ ]:
# 2. Скачивание датасета из Roboflow (при экспорте выберите формат: YOLO Darknet)
%cd /content/darknet
!pip install -q roboflow
from roboflow import Roboflow

# ВСТАВЬТЕ ВАШИ ДАННЫЕ ИЗ ROBOFLOW НИЖЕ:
# =======================СЮДА========================= #

# ==================================================== #
dataset_dir = dataset.location
print(f"Датасет скачан в: {dataset_dir}")

In [ ]:
# 3. Автоматическая генерация конфига yolov4-tiny-custom.cfg и файлов разметки
import os, glob, shutil

%cd /content/darknet
os.makedirs("backup", exist_ok=True)

# Читаем классы из _darknet.labels
labels_path = os.path.join(dataset_dir, "train", "_darknet.labels")
with open(labels_path, "r") as f:
    classes = [line.strip() for line in f.readlines() if line.strip()]
num_classes = len(classes)
print(f"Найдено классов ({num_classes}): {classes}")

shutil.copy(labels_path, "data/obj.names")

# Создаем train.txt и valid.txt
for split, out_file in [("train", "data/train.txt"), ("valid", "data/valid.txt")]:
    split_dir = os.path.join(dataset_dir, split)
    if not os.path.exists(split_dir) and split == "valid":
        split_dir = os.path.join(dataset_dir, "train")
    imgs = glob.glob(os.path.join(split_dir, "*.jpg")) + glob.glob(os.path.join(split_dir, "*.png"))
    with open(out_file, "w") as f:
        f.write("\n".join(imgs) + "\n")

# Создаем data/obj.data
with open("data/obj.data", "w") as f:
    f.write(f"classes = {num_classes}\n")
    f.write("train = data/train.txt\n")
    f.write("valid = data/valid.txt\n")
    f.write("names = data/obj.names\n")
    f.write("backup = backup/\n")

# Настраиваем cfg/yolov4-tiny-custom.cfg
max_batches = max(6000, num_classes * 2000)
step1 = int(max_batches * 0.8)
step2 = int(max_batches * 0.9)
filters = (num_classes + 5) * 3

with open("cfg/yolov4-tiny-custom.cfg", "r") as f:
    cfg = f.read()

cfg = cfg.replace("batch=1", "batch=64")
cfg = cfg.replace("subdivisions=1", "subdivisions=16")
cfg = cfg.replace("max_batches = 500200", f"max_batches = {max_batches}")
cfg = cfg.replace("steps=400000,450000", f"steps={step1},{step2}")
cfg = cfg.replace("classes=80", f"classes={num_classes}")
cfg = cfg.replace("filters=255", f"filters={filters}")
# Опционально: для ультра-легкой 'nano' модели можно уменьшить разрешение с 416 до 320:
# cfg = cfg.replace("width=416", "width=320").replace("height=416", "height=320")

with open("cfg/yolov4-tiny-custom.cfg", "w") as f:
    f.write(cfg)
print("Конфигурация успешно создана!")

In [ ]:
# 4. Запуск обучения
%cd /content/darknet
!./darknet detector train data/obj.data cfg/yolov4-tiny-custom.cfg yolov4-tiny.conv.29 -dont_show -map

In [ ]:
# 5. Скачивание обученных весов и конфига на компьютер
from google.colab import files
import os

best_weights = "/content/darknet/backup/yolov4-tiny-custom_best.weights"
last_weights = "/content/darknet/backup/yolov4-tiny-custom_last.weights"

if os.path.exists(best_weights):
    files.download(best_weights)
elif os.path.exists(last_weights):
    files.download(last_weights)

files.download("/content/darknet/cfg/yolov4-tiny-custom.cfg")
files.download("/content/darknet/data/obj.names")